# Visualising SAR and Optical timeseries Data Together

**Overview**

- Create an animation (.gif) showing a year of Sentinel-1 SAR data over the publications ice shelf in Antarctica.
- Compare this to Sentinel-2 optical data for the same time period
- Create a side-by-side animation for optical/SAR using a shared timeseries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import geopandas as gpd
import pandas as pd
import shapely
import xarray as xr
from PIL import Image, ImageSequence
from pystac_client import Client
import planetary_computer
from odc.stac import load, configure_s3_access
from odc.geo import BoundingBox
from de_sar_demo.speckle_filters import apply_lee_filter
from dea_tools.plotting import xr_animation

# 1. Sentinel-1 SAR Normalised Radar Backscatter (NRB)

### Environment set up
- Get data from the GA STAC endpoint

In [ ]:
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

### Settings for the STAC Query

In [ ]:
collection_name = "ga_s1_nrb_iw_hh_0" ## HH is the dominant polarisation of Antarctica
collection = stac_client.get_collection(collection_name)
collection

In [ ]:
# Area of interest - Publications Ice Shelf
aoi_bbox = BoundingBox(
    left=74.79566,
    bottom=-69.757,
    right=75.59814,
    top=-69.413,
    crs="EPSG:4326"
)

aoi_shape = shapely.geometry.shape(aoi_bbox.polygon)

# Date range of interest
start_date = "2019-01-01"
end_date = "2020-01-01"

#aoi_bbox.explore() # uncomment to show area on map

### Search for STAC Items

Note that the search function takes:
* `collections`: a list of collections.
* `datetime`: a date range in the format `"{start}/{end}"`.
* `intersects`: a geometry to intersect.

See the [API](https://pystac-client.readthedocs.io/en/latest/api.html#pystac_client.Client.search) for more information.

In [ ]:
items = stac_client.search(
    collections=[collection_name],
    datetime=f"{start_date}/{end_date}",
    intersects=aoi_bbox.boundary(),
).item_collection()

print(f"Found {len(items)} items")

print(f'STAC properties we are able to filter on:')
print(list(items[0].properties.keys()))

### Engineering : Limit data to relative orbits that completely cover the area of interest
- This ensures we have a complete SAR image for our area on any given day or satellite pass
- Without this filter, for any given day we may have partial coverage of an area which is not ideal for plotting

In [ ]:
# Convert STAC items to a GeoDataFrame
gdf = gpd.GeoDataFrame(
    [
        {
            "relative_orbit": item.properties["sat:relative_orbit"],
            "geometry": shapely.geometry.shape(item.geometry),
        }
        for item in items
    ],
    crs="EPSG:4326",
)

# Union geometries per relative orbit
gdf_union = gdf.dissolve(by="relative_orbit").reset_index()

# AOI geometry
aoi_gdf = gpd.GeoDataFrame(
    [{"geometry": aoi_shape}], crs="EPSG:4326"
)

# Plot showing the overlap between relative orbit and 
fig, ax = plt.subplots(figsize=(8, 8))
gdf_union.plot(ax=ax, legend=True, alpha=0.4, edgecolor="k",column="relative_orbit",categorical=True,)
aoi_gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2, label='AOI')
ax.set_title("Relative Orbit Coverage vs AOI", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.get_legend().set_title("Relative Orbit")
plt.show()

### Engineering : Limit the items for our orbit of interest

In [ ]:
items_to_load = [i for i in items if i.properties['sat:relative_orbit'] == 3]
print(f'Number of items to load : {len(items_to_load)}')

### Settings for STAC load

- Load the items into an xarray, and combine the individual bursts for a seamless dataset
- We can group by the flag `"solar_day"`, which combines all observations captured on a single day into one time-step in the loaded xarray.
- Alternatively, we can also group by the `"sat:absolute_orbit"` which will combine data for every unique pass. We know from out previous assessment that this should produce a complete picture of our area of interest for any given day.

In [ ]:
# Assets to load
assets_to_load = ["HH_gamma0"]

# CRS and resolution
output_crs = "EPSG:3031"
output_res = 50 # When loading a larger area, you can save time and memory by increasing the pixel size (lower resolution)

# Property or function to group by
groupy_by_operation = "sat:absolute_orbit" # "solar_day" will produce the same result

### Load the Items from STAC into xarray

Note that the load function takes:
* `items`: a pystac-client items_collection
* `bands`: a list of assets to load
* `geopolygon`: a geometry to be contained in
* `crs`: a coordinate reference system for the output
* `resolution`: a resolution for the output
* `groupby`: a method or property to group by
* `chunks`: (optionally) a dictionary describing how to chunk the data for lazy loading (recommended)

See the [API](https://odc-stac.readthedocs.io/en/latest/_api/odc.stac.load.html) for more information.

In [ ]:
ds_s1 = load(
    items=items_to_load,
    bands=assets_to_load,
    geopolygon=aoi_shape, # load for the shape
    crs=output_crs,
    resolution=output_res,
    groupby=groupy_by_operation,
    chunks={},
    all_touched=True,  # Only pixels fully inside geom
)
ds_s1.sortby("time")
print(f"Timesteps Found within Geometry: {len(ds_s1.time)}")

### Engineering : Speckle filter and convert to db for plotting

In [ ]:
ds_s1["HH_gamma0_filtered"] = apply_lee_filter(ds_s1.HH_gamma0, size=5)
ds_s1["HH_gamma0_filtered_db"] = 10*np.log10(ds_s1.HH_gamma0_filtered)
# ds_s1.isel(time=1).HH_gamma0_filtered_db.plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(6,5)) #uncomment to show sample plotted

# 2. Sentinel-2 Optical Surface Reflectance (SR)

### Envrionment Setup
- Get data from microsoft planetary computer STAC endpoint

In [ ]:
catalog = " https://planetarycomputer.microsoft.com/api/stac/v1"
stac_client = Client.open(catalog,modifier=planetary_computer.sign_inplace)

### Search for STAC Items

In [ ]:
s2_items = stac_client.search(
    collections=["sentinel-2-l2a"],
    datetime=f"{start_date}/{end_date}",
    intersects=aoi_bbox.boundary(),
).item_collection()
print(f"Sentinel-2 Items Found INTERSECTING the area : {len(s2_items)}")

### Load the Items from STAC into xarray 
- Only load STAC items with data in our AOI

In [ ]:
bands = ['B04','B03','B02'] #rgb

ds_s2 = load(
    s2_items,
    crs="EPSG:3031",
    resolution=50,
    geopolygon=aoi_shape,
    bands=bands,
    group_by="solar_day",
    chunks={},
    all_touched=False  # Only pixels fully inside geom
)
ds_s2.sortby("time")
print(f"Sentinel-2 Items Found within Geometry: {len(ds_s2.time)}")

# 3. Visualise optical and SAR side-by-side with same timeseries
- The Sentinel-1 and Sentinel-2 data is captures are on different dates, and the winter months of Sentinel-2 are missing.
- We Want to align the timeseries to get a complete picture across a full year:
  - In the summer months, fill the radar and optical timeseries with the nearest image to align the timeseries
  - In the winter months, set the optical image to be black as there is no data
  - Duplicate some data in the winter to increase sample density so months in animation are same length

In [ ]:
# Step 1: Compute the union of all times from both datasets
# Ensure time is sorted
ds_s1 = ds_s1.sortby("time")
ds_s2 = ds_s2.sortby("time")  # optional, if you use ds_s2 similarly
all_times = np.sort(np.unique(np.concatenate([ds_s1.time.values, ds_s2.time.values])))

# Step 2: Reindex SAR data using nearest neighbor
ds1_aligned = ds_s1.reindex(time=all_times, method="nearest")

# Step 3: Prepare empty data and time limits
empty_data = xr.zeros_like(ds_s2.isel(time=0)).expand_dims(time=[0])
before_date = np.datetime64("2019-04-03")
after_date = np.datetime64("2019-09-10")

# Step 4: Align optical data with black filling
ds2_aligned = xr.Dataset(coords={"time": all_times})
for var in ds_s2.data_vars:
    var_interp = ds_s2[var].interp(time=all_times, method="nearest")
    time_mask = xr.DataArray(all_times, dims="time")
    before_mask = time_mask < before_date
    after_mask = time_mask > after_date
    between_mask = ~before_mask & ~after_mask
    var_filled = var_interp.where(before_mask | after_mask, other=empty_data[var].isel(time=0))
    ds2_aligned[var] = var_filled
ds2_aligned.attrs.update(ds_s2.attrs)

# Step 5: Double points in winter months (e.g., June, July, August) for closer sample density across months
winter_months = [4, 5 ,6, 7, 8]
new_times = []
for i,t in enumerate(all_times):
    month = pd.Timestamp(t).month
    new_times.append(t)
    if month in winter_months:
        # Add duplicate timestamp (tiny offset to keep it unique)
        new_times.append(t + np.timedelta64(1, 'ms'))

# Step 6: Reindex both datasets to include duplicated times
ds1_winter_doubled = ds1_aligned.reindex(time=np.array(new_times), method="nearest")
ds2_winter_doubled = ds2_aligned.reindex(time=np.array(new_times), method="nearest")

# Both datasets share the same timesteps, with winter months duplicated
print(f'{len(ds1_winter_doubled.time)}')
print(f'{len(ds2_winter_doubled.time)}')

### Make animation for new SAR timeseries

In [ ]:
xr_animation(
    ds1_winter_doubled,
    bands="HH_gamma0_filtered_db",
    output_path='sentinel-1_timeseries_matched_2019_glacier_movement.gif',
    width_pixels=600,
    interval=80,
    show_date='%b %Y',
    show_text=None,
    show_colorbar=True,
    imshow_kwargs={"cmap":"bone", "vmin":-20, "vmax":0,"origin":"upper"},
    colorbar_kwargs={},
)

### Make animation for new Optical timeseries

In [ ]:
xr_animation(
    ds2_winter_doubled,
    bands=bands,
    output_path='sentinel-2_timeseries_matched_2019_glacier_movement.gif',
    width_pixels=600,
    interval=80,
    show_date='%b %Y',
    show_text=None,
    show_colorbar=False,
    colorbar_kwargs={},
)

### Combine the SAR and Optical GIFS to be side by slide

In [ ]:
def uniform_frames(gif_path, frame_interval=100):
    """Convert GIF to a list of frames at uniform intervals (ms). As there are
    'empty' repeated frames in the winter months of the .gif, we need to ensure
    the frames and intervals are consistent"""
    gif = Image.open(gif_path)
    frames_list = [frame.convert("RGBA") for frame in ImageSequence.Iterator(gif)]
    
    # Original frame durations
    durations = [frame.info.get('duration', gif.info['duration']) for frame in ImageSequence.Iterator(gif)]
    cum_times = np.cumsum(durations)
    total_time = cum_times[-1]
    
    # Number of uniform frames
    times = np.arange(0, total_time, frame_interval)
    
    uniform_frames = []
    f_idx = 0
    for t in times:
        # Advance to correct frame for this time
        while f_idx < len(cum_times) - 1 and t >= cum_times[f_idx]:
            f_idx += 1
        uniform_frames.append(frames_list[f_idx])
    
    return uniform_frames

# Load uniform frames
frames1 = uniform_frames("sentinel-2_timeseries_matched_2019_glacier_movement.gif", frame_interval=80)
frames2 = uniform_frames("sentinel-1_timeseries_matched_2019_glacier_movement.gif", frame_interval=80)

# Combine side by side
combined_frames = []
for f1, f2 in zip(frames1, frames2):
    new_frame = Image.new("RGBA", (f1.width + f2.width, f1.height))
    new_frame.paste(f1, (0, 0))
    new_frame.paste(f2, (f1.width, 0))
    combined_frames.append(new_frame)

# Save combined GIF
combined_frames[0].save(
    "SAR_optical_2019_glacier_movement.gif",
    save_all=True,
    append_images=combined_frames[1:],
    duration=80,  # uniform frame duration
    loop=0
)
combined_frames[0]